# Session 5 — Agent RL & Knowledge Distillation

**Duration:** 75 min  
**Repository:** [MiniMind-Colab](https://github.com/your-org/minimind-colab)

| Part | Topic | Time |
|------|-------|------|
| A | Agent RL for Tool Use | 40 min |
| B | Knowledge Distillation | 30 min |
| — | Wrap-up & Q\&A | 5 min |

In [ ]:
# Cell — Environment setup
!pip install torch transformers datasets --quiet

import math, json, os, time, random, re
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch import optim
from torch.utils.data import Dataset, DataLoader
from contextlib import nullcontext
from transformers.activations import ACT2FN
from transformers import PreTrainedModel, GenerationMixin, PretrainedConfig, AutoTokenizer
from transformers.modeling_outputs import MoeCausalLMOutputWithPast

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch {torch.__version__}  CUDA available: {torch.cuda.is_available()}')
print(f'Device: {device}')

## Part A — Agent RL for Tool Use (40 min)

Agent RL teaches a language model to **use external tools** by generating
structured `<tool_call>` XML blocks. The model learns which tool to invoke
and what arguments to pass through reinforcement-learning rewards.

### Key ideas

1. **Tool definitions** describe callable functions (name, parameters).
2. The model generates `<tool_call>{"name": ..., "arguments": ...}</tool_call>` blocks.
3. A **mock executor** runs the tool and returns a JSON result.
4. A **reward function** scores whether the correct tool was called with the right arguments.
5. Multi-turn rollouts let the model chain tool calls before giving a final answer.

### Prerequisites from Sessions 1–4

We re-define the model architecture from previous sessions.
These are provided — no changes needed.

In [ ]:
# Cell — Prerequisites: Model architecture (from Sessions 1–4)

class RMSNorm(torch.nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))
    def norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
    def forward(self, x):
        return (self.weight * self.norm(x.float())).type_as(x)

def precompute_freqs_cis(dim, end=32*1024, rope_base=1e6):
    freqs = 1.0 / (rope_base ** (torch.arange(0, dim, 2)[:(dim // 2)].float() / dim))
    t = torch.arange(end)
    freqs = torch.outer(t, freqs)
    return torch.polar(torch.ones_like(freqs), freqs)

def apply_rotary_emb(xq, xk, freqs_cis):
    xq_ = torch.view_as_complex(xq.float().reshape(*xq.shape[:-1], -1, 2))
    xk_ = torch.view_as_complex(xk.float().reshape(*xk.shape[:-1], -1, 2))
    freqs_cis = freqs_cis[:xq_.shape[1]].unsqueeze(0).unsqueeze(2)
    xq_out = torch.view_as_real(xq_ * freqs_cis).flatten(3)
    xk_out = torch.view_as_real(xk_ * freqs_cis).flatten(3)
    return xq_out.type_as(xq), xk_out.type_as(xk)

class Attention(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.hidden_size = config.hidden_size
        self.num_heads = config.num_attention_heads
        self.num_kv_heads = config.num_key_value_heads
        self.head_dim = config.head_dim
        self.num_kv_groups = self.num_heads // self.num_kv_heads
        self.q_proj = nn.Linear(self.hidden_size, self.num_heads * self.head_dim, bias=False)
        self.k_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.v_proj = nn.Linear(self.hidden_size, self.num_kv_heads * self.head_dim, bias=False)
        self.o_proj = nn.Linear(self.num_heads * self.head_dim, self.hidden_size, bias=False)

    def forward(self, x, freqs_cis, attention_mask=None, past_kv=None, use_cache=False):
        B, T, _ = x.shape
        q = self.q_proj(x).view(B, T, self.num_heads, self.head_dim)
        k = self.k_proj(x).view(B, T, self.num_kv_heads, self.head_dim)
        v = self.v_proj(x).view(B, T, self.num_kv_heads, self.head_dim)
        q, k = apply_rotary_emb(q, k, freqs_cis)
        if past_kv is not None:
            pk, pv = past_kv
            k = torch.cat([pk, k], dim=1)
            v = torch.cat([pv, v], dim=1)
        new_kv = (k, v) if use_cache else None
        if self.num_kv_groups > 1:
            k = k.unsqueeze(3).expand(-1, -1, -1, self.num_kv_groups, -1).reshape(B, -1, self.num_heads, self.head_dim)
            v = v.unsqueeze(3).expand(-1, -1, -1, self.num_kv_groups, -1).reshape(B, -1, self.num_heads, self.head_dim)
        q, k, v = [t.transpose(1, 2) for t in (q, k, v)]
        attn = F.scaled_dot_product_attention(q, k, v, attn_mask=attention_mask, is_causal=(attention_mask is None and T > 1))
        out = attn.transpose(1, 2).contiguous().view(B, T, -1)
        return self.o_proj(out), new_kv

class FeedForward(nn.Module):
    def __init__(self, config):
        super().__init__()
        hidden = int(config.hidden_size * 8 / 3)
        hidden = ((hidden + 63) // 64) * 64
        self.gate_proj = nn.Linear(config.hidden_size, hidden, bias=False)
        self.up_proj = nn.Linear(config.hidden_size, hidden, bias=False)
        self.down_proj = nn.Linear(hidden, config.hidden_size, bias=False)
        self.act_fn = ACT2FN[config.hidden_act]
    def forward(self, x):
        return self.down_proj(self.act_fn(self.gate_proj(x)) * self.up_proj(x))

class TransformerBlock(nn.Module):
    def __init__(self, layer_id, config):
        super().__init__()
        self.attention = Attention(config)
        self.feed_forward = FeedForward(config)
        self.attention_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
    def forward(self, x, freqs_cis, attention_mask=None, past_kv=None, use_cache=False):
        h, new_kv = self.attention(self.attention_norm(x), freqs_cis, attention_mask, past_kv, use_cache)
        h = x + h
        out = h + self.feed_forward(self.ffn_norm(h))
        return out, new_kv

class MiniMindConfig(PretrainedConfig):
    model_type = 'minimind'
    def __init__(self, hidden_size=128, num_hidden_layers=2, num_attention_heads=4,
                 num_key_value_heads=2, head_dim=32, vocab_size=32000, max_seq_len=32768,
                 rms_norm_eps=1e-6, rope_theta=1e6, hidden_act='silu', use_moe=False, **kw):
        self.hidden_size = hidden_size
        self.num_hidden_layers = num_hidden_layers
        self.num_attention_heads = num_attention_heads
        self.num_key_value_heads = num_key_value_heads
        self.head_dim = head_dim or hidden_size // num_attention_heads
        self.vocab_size = vocab_size
        self.max_seq_len = max_seq_len
        self.rms_norm_eps = rms_norm_eps
        self.rope_theta = rope_theta
        self.hidden_act = hidden_act
        self.use_moe = use_moe
        super().__init__(**kw)

class MiniMindForCausalLM(PreTrainedModel, GenerationMixin):
    config_class = MiniMindConfig
    def __init__(self, config):
        super().__init__(config)
        self.tok_embeddings = nn.Embedding(config.vocab_size, config.hidden_size)
        self.layers = nn.ModuleList([TransformerBlock(i, config) for i in range(config.num_hidden_layers)])
        self.norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.output = nn.Linear(config.hidden_size, config.vocab_size, bias=False)
        self.register_buffer('freqs_cis',
            precompute_freqs_cis(config.head_dim, config.max_seq_len, config.rope_theta), persistent=False)
        self.aux_loss = torch.tensor(0.0)
        self.OUT = MoeCausalLMOutputWithPast

    def forward(self, input_ids=None, labels=None, attention_mask=None,
                past_key_values=None, use_cache=False, **kw):
        h = self.tok_embeddings(input_ids)
        new_kvs = []
        for i, layer in enumerate(self.layers):
            pk = past_key_values[i] if past_key_values else None
            h, nk = layer(h, self.freqs_cis, attention_mask, pk, use_cache)
            new_kvs.append(nk)
        logits = self.output(self.norm(h))
        loss = None
        if labels is not None:
            loss = F.cross_entropy(
                logits[:, :-1].reshape(-1, logits.size(-1)),
                labels[:, 1:].reshape(-1), ignore_index=-100
            )
        return self.OUT(loss=loss, logits=logits, aux_loss=self.aux_loss,
                        past_key_values=new_kvs if use_cache else None)

print('Model architecture defined ✓')

### 1. Tool Definitions & Mock Data

We define three tools the agent can call: a math calculator, a weather
lookup, and an exchange-rate query. Mock data lets us test locally
without real API calls.

In [ ]:
# Cell — Tool definitions and mock data (provided)

TOOLS = [
    {"type": "function", "function": {"name": "calculate_math", "description": "Calculate math expressions", "parameters": {"type": "object", "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}},
    {"type": "function", "function": {"name": "get_current_weather", "description": "Get weather", "parameters": {"type": "object", "properties": {"location": {"type": "string"}}, "required": ["location"]}}},
    {"type": "function", "function": {"name": "get_exchange_rate", "description": "Get exchange rate", "parameters": {"type": "object", "properties": {"from_currency": {"type": "string"}, "to_currency": {"type": "string"}}, "required": ["from_currency", "to_currency"]}}},
]

WEATHER_DATA = {"Beijing": ("28°C", "Sunny"), "New York": ("8°C", "Cloudy")}
EXCHANGE_DATA = {("USD", "CNY"): 7.21, ("EUR", "CNY"): 7.85}

MOCK_RESULTS = {
    "calculate_math": lambda args: {"result": str(eval(str(args.get("expression", "0")).replace("^", "**"), {"__builtins__": {}, "math": __import__('math')}))},
    "get_current_weather": lambda args: {"city": args.get("location"), "temperature": WEATHER_DATA.get(args.get("location"), ("22°C", "Sunny"))[0]},
    "get_exchange_rate": lambda args: {"rate": EXCHANGE_DATA.get((args.get("from_currency"), args.get("to_currency")), 1.0)},
}

print('Tool definitions loaded ✓')
print(f'Available tools: {[t["function"]["name"] for t in TOOLS]}')

### 2. Parsing Tool Calls

The model outputs tool invocations wrapped in XML-style tags:

```
<tool_call>{"name": "calculate_math", "arguments": {"expression": "2+3"}}</tool_call>
```

We need a function that extracts all such blocks from the generated text
and parses the JSON payload inside each one.

In [ ]:
# Cell — parse_tool_calls implementation

def parse_tool_calls(text):
    calls = []
    for m in re.findall(r'<tool_call>(.*?)</tool_call>', text, re.DOTALL):
        try:
            calls.append(json.loads(m.strip()))
        except:
            pass
    return calls

print('parse_tool_calls defined ✓')

### 3. Executing a Tool Call

Given a tool name and its arguments, look up the corresponding mock
function in `MOCK_RESULTS` and execute it. Return `None` if the tool
name is unknown or execution fails.

In [ ]:
# Cell — execute_tool implementation

def execute_tool(name, args):
    fn = MOCK_RESULTS.get(name)
    if not fn:
        return None
    try:
        return fn(args)
    except:
        return None

print('execute_tool defined ✓')

### ✅ Milestone 1: Tool Parsing & Execution

We verify that `parse_tool_calls` correctly extracts JSON from `<tool_call>` tags
and that `execute_tool` returns valid results for weather and math queries.

In [ ]:
# Cell — ✅ Milestone 1: Tool parsing & execution

# Test parse_tool_calls
sample_text = 'Let me check. <tool_call>{"name": "calculate_math", "arguments": {"expression": "2+3"}}</tool_call> Done.'
parsed = parse_tool_calls(sample_text)
print(f'Parsed calls: {parsed}')
assert len(parsed) == 1, f'Expected 1 parsed call, got {len(parsed)}'
assert parsed[0]['name'] == 'calculate_math', f'Wrong tool name: {parsed[0].get("name")}'
assert parsed[0]['arguments'] == {'expression': '2+3'}, f'Wrong arguments: {parsed[0].get("arguments")}'

# Test execute_tool — math
math_result = execute_tool('calculate_math', {'expression': '2+3'})
print(f'Math result: {math_result}')
assert math_result is not None, 'Math tool returned None'
assert math_result['result'] == '5', f'Expected "5", got {math_result["result"]}'

# Test execute_tool — weather
weather_result = execute_tool('get_current_weather', {'location': 'Beijing'})
print(f'Weather result: {weather_result}')
assert weather_result is not None, 'Weather tool returned None'
assert weather_result['temperature'] == '28°C', f'Expected 28°C, got {weather_result["temperature"]}'

# Test unknown tool
assert execute_tool('unknown_tool', {}) is None, 'Unknown tool should return None'

print('✅ Milestone 1 passed — parse_tool_calls and execute_tool work correctly')

### 4. Agent Reward Function

The reward function scores how well the model used tools:

| Component | Score |
|-----------|-------|
| Called any tool | +0.5 |
| Correct tool name | +1.0 |
| Correct arguments | +1.0 |
| Reasonable length (20–800 chars) | +0.5 / −0.5 |

The ground-truth `gt` dict contains the expected `name` and `arguments`.

In [ ]:
# Cell — agent_reward implementation

def agent_reward(response, gt):
    reward = 0.0
    calls = parse_tool_calls(response)
    if calls:
        reward += 0.5
        for call in calls:
            if call.get('name') == gt.get('name'):
                reward += 1.0
            if call.get('arguments') == gt.get('arguments'):
                reward += 1.0
    reward += 0.5 if 20 <= len(response.strip()) <= 800 else -0.5
    return reward

print('agent_reward defined ✓')

### ✅ Milestone 2: Agent Reward Scoring

We verify that `agent_reward` assigns correct scores for perfect,
partial, and missing tool calls.

In [ ]:
# Cell — ✅ Milestone 2: Agent reward scoring

gt = {'name': 'get_current_weather', 'arguments': {'location': 'Beijing'}}

# Perfect response
perfect = 'Let me check the weather. <tool_call>{"name": "get_current_weather", "arguments": {"location": "Beijing"}}</tool_call>'
r_perfect = agent_reward(perfect, gt)
print(f'Perfect response reward: {r_perfect}')
assert r_perfect >= 2.5, f'Perfect reward should be >= 2.5, got {r_perfect}'

# Wrong tool name
wrong_name = 'Here it is. <tool_call>{"name": "calculate_math", "arguments": {"location": "Beijing"}}</tool_call>'
r_wrong = agent_reward(wrong_name, gt)
print(f'Wrong-name reward: {r_wrong}')
assert r_wrong < r_perfect, f'Wrong-name reward ({r_wrong}) should be less than perfect ({r_perfect})'

# No tool call at all
no_tool = 'The weather in Beijing is nice today.'
r_none = agent_reward(no_tool, gt)
print(f'No-tool reward: {r_none}')
assert r_none <= 0.5, f'No-tool reward should be <= 0.5, got {r_none}'

print('✅ Milestone 2 passed — agent_reward scores correct, partial, and missing tool calls')

### 5. Multi-Turn Rollout (Provided)

The `rollout_single` function handles the agent’s multi-turn loop:
generate → parse tool calls → execute → append result → repeat.

This is provided as-is — it is complex and would take too long to
implement in a 75-minute session. Study it to understand the flow.

In [ ]:
# Cell — rollout_single (provided)

def rollout_single(model, tokenizer, messages, tools, max_turns=3, max_new_tokens=256, device='cuda'):
    """Multi-turn agent rollout: generate, parse tool calls, execute, loop."""
    for turn in range(max_turns):
        context = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, tools=tools
        )
        inputs = tokenizer(context, return_tensors='pt', add_special_tokens=False).to(device)
        output = model.generate(
            inputs['input_ids'], max_new_tokens=max_new_tokens,
            do_sample=True, temperature=0.8
        )
        new_text = tokenizer.decode(
            output[0][len(inputs['input_ids'][0]):], skip_special_tokens=True
        )
        calls = parse_tool_calls(new_text)
        if not calls:
            break
        messages.append({'role': 'assistant', 'content': new_text})
        for call in calls:
            name, raw = call.get('name', ''), call.get('arguments', {})
            result = execute_tool(name, raw)
            result_str = json.dumps(result, ensure_ascii=False) if result else '{"error": "tool not found"}'
            messages.append({'role': 'tool', 'content': result_str})
    return new_text, messages

print('rollout_single defined ✓')

## Part B — Knowledge Distillation (30 min)

Knowledge distillation transfers knowledge from a large **teacher** model
to a smaller **student** model by matching their output distributions.

### Key equation

$$\mathcal{L}_{\text{KD}} = T^2 \cdot \text{KL}\!\left(\text{softmax}\!\left(\frac{z_s}{T}\right) \,\|\, \text{softmax}\!\left(\frac{z_t}{T}\right)\right)$$

Where:
- $z_s$, $z_t$ are student and teacher logits
- $T$ is the **temperature** (higher $T$ softens distributions, revealing more structure)
- $T^2$ compensates for the reduced gradient magnitude at high temperatures

### Blended loss

The total training loss blends cross-entropy with ground truth and distillation:

$$\mathcal{L} = \alpha \cdot \mathcal{L}_{\text{CE}} + (1 - \alpha) \cdot \mathcal{L}_{\text{KD}}$$

### 6. Distillation Loss Function

Implement the KL-divergence-based distillation loss with temperature
scaling. The teacher’s soft targets are computed with `torch.no_grad()`
to avoid unnecessary gradient computation.

In [ ]:
# Cell — distillation_loss implementation

def distillation_loss(student_logits, teacher_logits, temperature=1.0, reduction='batchmean'):
    with torch.no_grad():
        teacher_probs = F.softmax(teacher_logits / temperature, dim=-1).detach()
    student_log_probs = F.log_softmax(student_logits / temperature, dim=-1)
    kl = F.kl_div(student_log_probs, teacher_probs, reduction=reduction)
    return (temperature ** 2) * kl

print('distillation_loss defined ✓')

### 7. Distillation Training Demo

We create a tiny teacher and student model, then run a short training loop
using the blended loss $\alpha \cdot \mathcal{L}_{\text{CE}} + (1 - \alpha) \cdot \mathcal{L}_{\text{KD}}$.

In [ ]:
# Cell — Distillation training demo

# Create teacher (larger) and student (smaller)
teacher_config = MiniMindConfig(
    hidden_size=128, num_hidden_layers=4, num_attention_heads=4,
    num_key_value_heads=4, head_dim=32, vocab_size=6400,
)
student_config = MiniMindConfig(
    hidden_size=64, num_hidden_layers=2, num_attention_heads=2,
    num_key_value_heads=2, head_dim=32, vocab_size=6400,
)

torch.manual_seed(42)
teacher_model = MiniMindForCausalLM(teacher_config).to(device)
student_model = MiniMindForCausalLM(student_config).to(device)
teacher_model.eval()
teacher_model.requires_grad_(False)

print(f'Teacher params: {sum(p.numel() for p in teacher_model.parameters()) / 1e3:.1f}K')
print(f'Student params: {sum(p.numel() for p in student_model.parameters()) / 1e3:.1f}K')

# Training hyperparameters
ALPHA = 0.5
TEMPERATURE = 2.0
LR = 1e-3
NUM_STEPS = 30
B, T, V = 4, 32, 6400

optimizer = optim.AdamW(student_model.parameters(), lr=LR)

losses = []
for step in range(NUM_STEPS):
    input_ids = torch.randint(0, V, (B, T)).to(device)
    labels = torch.randint(0, V, (B, T)).to(device)

    # Student forward
    student_out = student_model(input_ids)
    student_logits = student_out.logits[:, :-1, :].contiguous()
    shift_labels = labels[:, 1:].contiguous()

    # CE loss
    ce_loss = F.cross_entropy(
        student_logits.view(-1, V), shift_labels.view(-1), reduction='mean'
    )

    # Teacher forward
    with torch.no_grad():
        teacher_logits = teacher_model(input_ids).logits[:, :-1, :].contiguous()
        teacher_logits = teacher_logits[..., :V]

    # Distillation loss
    distill_loss = distillation_loss(
        student_logits.view(-1, V),
        teacher_logits.view(-1, V),
        temperature=TEMPERATURE,
    )

    # Blended loss
    loss = ALPHA * ce_loss + (1 - ALPHA) * distill_loss

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    if step % 10 == 0:
        print(f'Step {step:3d}  loss={loss.item():.4f}  ce={ce_loss.item():.4f}  kd={distill_loss.item():.4f}')

print(f'\nTraining complete. Final loss: {losses[-1]:.4f}')

### ✅ Milestone 3: Distillation Loss Verification

We verify that `distillation_loss` produces a valid positive scalar that is not NaN.

In [ ]:
# Cell — ✅ Milestone 3: Distillation loss verification

# Test with synthetic logits
torch.manual_seed(42)
test_student_logits = torch.randn(8, 6400).to(device)
test_teacher_logits = torch.randn(8, 6400).to(device)

test_loss = distillation_loss(test_student_logits, test_teacher_logits, temperature=2.0)
print(f'Distillation loss value: {test_loss.item():.4f}')
print(f'Is scalar: {test_loss.dim() == 0}')
print(f'Is positive: {test_loss.item() > 0}')
print(f'Is not NaN: {not torch.isnan(test_loss).item()}')

assert not torch.isnan(test_loss), 'Distillation loss is NaN!'
assert test_loss.item() > 0, f'Distillation loss should be positive, got {test_loss.item()}'
assert test_loss.dim() == 0, f'Loss should be a scalar, got shape {test_loss.shape}'

# Test that higher temperature gives different (typically lower) loss
loss_t1 = distillation_loss(test_student_logits, test_teacher_logits, temperature=1.0)
loss_t4 = distillation_loss(test_student_logits, test_teacher_logits, temperature=4.0)
print(f'Loss at T=1.0: {loss_t1.item():.4f}')
print(f'Loss at T=4.0: {loss_t4.item():.4f}')
assert not torch.isnan(loss_t1) and not torch.isnan(loss_t4), 'Loss is NaN at different temperatures!'

print('✅ Milestone 3 passed — distillation_loss returns a valid positive scalar, not NaN')

## Wrap-up (5 min)

### What we built today

| Component | Key idea |
|-----------|----------|
| **parse_tool_calls** | Regex extraction of JSON from `<tool_call>` XML tags |
| **execute_tool** | Look up mock function by name, execute with args |
| **agent_reward** | Score tool-use quality: correct name, arguments, length |
| **distillation_loss** | Temperature-scaled KL divergence with T² compensation |

### Training pipeline

```
Pretrain → SFT → LoRA → DPO / GRPO → Agent RL + Distillation
                                       ↑ Session 5
```

### Key takeaways
- **Agent RL** teaches models to call tools via structured XML; rewards guide correct tool selection.
- **Multi-turn rollouts** let the agent chain calls before giving a final answer.
- **Knowledge distillation** compresses a large teacher into a smaller student by matching soft distributions.
- The **temperature** parameter controls how much of the teacher’s distribution structure is transferred.
- Blending CE and KD losses ($\alpha$) balances ground-truth fit with teacher imitation.